In [1]:
import torch
import torchvision
import requests
import json
from torchvision.io import read_video
from torchvision.models.video import r3d_18, R3D_18_Weights
import glob 
import pandas

In [2]:
test_list_pass = glob.glob('./data/test/no_shot/*.mp4')
labels_test_pass = [0 for i in range(len(test_list_pass))]

test_list_shot = glob.glob('./data/test/shot/*.mp4')
labels_test_shot = [1 for i in range(len(test_list_shot))]

videos_test = test_list_pass + test_list_shot
labels_test = labels_test_pass + labels_test_shot

print(f'Test videos {len(videos_test)}')

Test videos 68


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

print("Loading pre-trained ResNet3D-18 model")
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model = model.to(DEVICE)
model.eval()
kinetics_labels = weights.meta["categories"]
preprocess = weights.transforms()

Using device: cuda
Loading pre-trained ResNet3D-18 model


In [ ]:
for VIDEO_PATH in videos_test:
    print(f"\nLoading video from: {VIDEO_PATH}")
    try:
        # 1. Load the video -> returns (T, C, H, W)
        video_frames, audio, info = read_video(VIDEO_PATH, output_format="TCHW")
        print(f"Original Video shape: {video_frames.shape}")
        
        # 2. Scale pixel values
        video_frames = video_frames.float() / 255.0
        
        # 3. Temporal Sampling (Still in T, C, H, W format here)
        num_frames = video_frames.shape[0]  # T is index 0
        desired_frames = 16
        
        if num_frames >= desired_frames:
            indices = torch.linspace(0, num_frames - 1, steps=desired_frames).long()
            video_sampled = video_frames[indices]
        else:
            # Pad if the video is extremely short
            padding_size = desired_frames - num_frames
            last_frame = video_frames[-1:]
            padding = last_frame.repeat(padding_size, 1, 1, 1)
            video_sampled = torch.cat([video_frames, padding], dim=0)

        print(f"Sampled Video shape (T, C, H, W): {video_sampled.shape}")
        print("\nPreprocessing video...")

        # 4. Torchvision transform handles spatial crops AND permutes to (C, T, H, W)
        video_preprocessed = preprocess(video_sampled)
        print(f"Preprocessed video shape: {video_preprocessed.shape}")
        
        # 5. Add Batch dimension -> (B, C, T, H, W)
        if video_preprocessed.dim() == 4:
            video_preprocessed = video_preprocessed.unsqueeze(0)
        
        print("\nRunning inference...")
        with torch.no_grad():
            outputs = model(video_preprocessed.to(DEVICE))
        
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        top1_probs, top1_indices = torch.topk(probabilities, 1, dim=1)
        print(f'Class "{kinetics_labels[top1_indices.item()]}" with probability {top1_probs.item() * 100:.2f}%')

    except FileNotFoundError:
        print(f"Error: Video file '{VIDEO_PATH}' not found!")
    except Exception as e:
        print(f"Error during processing: {type(e).__name__}: {e}")


Loading video from: ./data/test/no_shot\107.mp4


c:\Users\hecto\anaconda3\envs\video_inference\lib\site-packages\torchvision\io\video.py:169: UserWarning: The pts_unit 'pts' gives wrong results. Please use pts_unit 'sec'.
  warnings.warn("The pts_unit 'pts' gives wrong results. Please use pts_unit 'sec'.")


Original Video shape: torch.Size([417, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])

Preprocessing video...
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Running inference...
Class "dunking basketball" with probability 75.52%

Loading video from: ./data/test/no_shot\116.mp4
Original Video shape: torch.Size([471, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])

Preprocessing video...
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Running inference...
Class "dunking basketball" with probability 90.25%

Loading video from: ./data/test/no_shot\121.mp4
Error during processing: AttributeError: module 'av' has no attribute 'AVError'

Loading video from: ./data/test/no_shot\133.mp4
Original Video shape: torch.Size([513, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])

Preprocessing video...
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Running inference...
Class "d